<h3 style="color:#99ffcc"> Data preprocessing and machine learning</h3> 

In [2]:
import pandas as pd
import numpy as np
import sys
import matplotlib.pyplot as plt
from scipy.stats import linregress, logistic, kstest
from scipy.optimize import curve_fit
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, r2_score, mean_squared_error

- Program uses data from the dataset available at https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset

- First, we cleaned the data by removing patients with "Unknown" values. We also dropped the unnecessary ID column

- After that, mappings were created for each variable in the dataset. For variables with only two possible values, we assigned the value $0$ to the first option and the value $1$ to the second option

- For all options in the <i> smoking_status</i>  and  <i>work_type</i> variables, zeros were assigned

- Maps with the name containing <i>input</i> phrase are used later in the <text style="color:#99ffcc"> profile()</text> function. The rest are used to prepare data to the machine learning process

- Variables <i>gender, ever_married, Residence_type</i> were changed to the numeric values according to their maps

- Columns for the <i>smoking_status</i> and the <i>work_type</i> were split by using <text style="color:#99ffcc"> pd.get_dummies </text>. This variables are now represented by i-dimensional vectors, where $i$ is the number of possible values ($5$ for the <i>work_type</i> and $3$ for the <i>smoking_status</i>)

In [3]:
# Data preprocessing
dataset = pd.read_csv('healthcare-dataset-stroke-data.csv').drop(columns=['id']).dropna()         # Loading dataset without id column and missing values
dataset = dataset[dataset['gender'] != 'Other']
dataset = dataset[dataset['smoking_status'] != 'Unknown']

gender_mapping = {'Male': 0, 'Female': 1}                                                         # Mapping necessary values
married_mapping = {'No': 0, 'Yes': 1}
residence_mapping = {'Rural': 0, 'Urban': 1}
input_employment_mapping = {'Private': 0, 'Self-employed': 0, 'Children': 0, 'Government job': 0, 'Never worked': 0}
input_smoking_mapping = {'Never smoked': 0, 'Formerly smoked': 0, 'Smokes': 0} 
input_hypertension_mapping = {'No': 0, 'Yes': 1}
input_heart_disease_mapping = {'No': 0, 'Yes': 1}
input_ever_married_mapping = {'No': 0, 'Yes': 1}
input_residence_mapping = {'Rural': 0, 'Urban': 1}

dataset['gender'] = dataset['gender'].map(gender_mapping)
dataset['ever_married'] = dataset['ever_married'].map(married_mapping)
dataset['Residence_type'] = dataset['Residence_type'].map(residence_mapping)

dataset = pd.get_dummies(dataset, columns=['work_type'], dtype=int)
dataset = pd.get_dummies(dataset, columns=['smoking_status'], dtype=int)

dataset

,gender,age,hypertension,heart_disease,ever_married,Residence_type,avg_glucose_level,bmi,stroke,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,0,67.0,0,1,1,1,228.69,36.6,1,0,0,1,0,0,1,0,0
2,0,80.0,0,1,1,0,105.92,32.5,1,0,0,1,0,0,0,1,0
3,1,49.0,0,0,1,1,171.23,34.4,1,0,0,1,0,0,0,0,1
4,1,79.0,1,0,1,0,174.12,24.0,1,0,0,0,1,0,0,1,0
5,0,81.0,0,0,1,1,186.21,29.0,1,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5100,0,82.0,1,0,1,0,71.97,28.3,0,0,0,0,1,0,0,1,0
5102,1,57.0,0,0,1,0,77.93,21.7,0,0,0,1,0,0,0,1,0
5106,1,81.0,0,0,1,1,125.20,40.0,0,0,0,0,1,0,0,1,0
5107,1,35.0,0,0,1,0,82.99,30.6,0,0,0,0,1,0,0,1,0


- Dataset was divided into <i>training_set</i>, containing $80\%$ of the original data, and <i>testing_set</i>. Column <text style="color:#99ffcc"> "stroke" </text> is stored separately for each set to avoid data leakage

- <text style="color:#99ffcc">StandardScaler()</text> function was used for standarization of non-binary data in both sets. Standarization was needed to avoid errors connected with the variable scales during the machine learning process

- <text style="color:#99ffcc">StandardScaler().transform</text> was used for the testing set insead of the <text style="color:#99ffcc">StandardScaler().fit_transform</text> to avoid data leakage

In [4]:
# Dataset divisions
data = dataset.drop(columns=['stroke'])
result = dataset['stroke']
training_set, testing_set, training_result, testing_result = train_test_split(data, result, test_size=0.2, random_state=42)
scaler = StandardScaler()
cols_to_scale = ['age', 'avg_glucose_level', 'bmi']
training_set[cols_to_scale] = scaler.fit_transform(training_set[cols_to_scale])
testing_set[cols_to_scale] = scaler.transform(testing_set[cols_to_scale])

training_set

,gender,age,hypertension,heart_disease,ever_married,Residence_type,avg_glucose_level,bmi,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
4509,0,-1.746462,0,0,0,1,-0.537871,-1.235804,0,0,1,0,0,0,1,0
4302,0,-0.997085,0,0,0,1,-0.950542,-0.311202,0,0,1,0,0,0,1,0
1113,1,0.929884,0,0,1,0,3.022762,0.697455,0,0,0,1,0,0,0,1
3787,1,-1.639408,0,0,0,1,-0.264870,-1.011658,0,0,1,0,0,0,1,0
4909,1,-0.515343,0,0,1,1,2.854989,0.977637,0,0,1,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1593,0,-0.943558,0,0,1,1,-0.783825,-0.619402,0,0,1,0,0,0,1,0
1653,0,-0.247709,0,0,1,0,-0.289381,-0.255165,0,0,1,0,0,0,0,1
1899,1,-1.157666,0,0,0,0,-0.531320,-1.123731,0,0,1,0,0,0,1,0
1236,0,0.983411,0,0,1,0,-0.310511,0.137090,1,0,0,0,0,1,0,0


- The <text style="color:#99ffcc"> logistic regression</text> was chosen for the machine learning model

- After machine learning process, an evaluation was conducted. Results were significantly better than for XGBoost model, which was implemented earlier

In [5]:
# Machine learning model (Logistic Regression)
model = LogisticRegression(class_weight='balanced', random_state=42)
model.fit(training_set, training_result)

# Evaluation
predictions = model.predict(testing_set)

<h3 style="color:#99ffcc"> User's profile creation</h3> 

- Firstly, some supplementary functions were defined to check if the user input has correct format and is in the desirable range

In [6]:
def get_integer_input(prompt: str, range_min = 0, range_max = sys.maxsize - 1):
    while True:
        while not (age := input(f"{prompt}:")).isdigit():
            print("Please provide numeric value \n")
        age = int(age)
        if age < range_min or age > range_max:
            print(f"Please provide value between: {range_min} and {range_max}")
        else:
            return age


def get_float_input(prompt: str, range_min = 0, range_max = sys.maxsize - 1):
    while True:
        try:
            feature = float(input(f"{prompt} "))
            if feature < range_min or feature > range_max:
                print(f"Please provide value between: {range_min} and {range_max}")
            else:
                return feature
        except ValueError:
            print("Please provide numeric value")


def get_generic_input(prompt: str, options: tuple):
    while (user_input := input(f"{prompt} ({'/'.join(options)}):").lower()) not in [option.lower() for option in options]:
        print("Please provide a valid option")
    first_letter = user_input[0].upper()
    user_input = user_input[1:]
    user_input = first_letter + user_input
    return user_input


- After that, the function <text style="color:#99ffcc">profile()</text> was defined

- All data entered by the user was converted with mapping to match the format of the training set

- If user's age is above $17$, <text style="color:#99ffcc">valid_work_types.pop('Children')</text> guarantees that option <i>'Children'</i> can't be chosen within <i>work_type</i> variable

- Variables <i>age, avg_glucose_level</i> and <i>bmi</i> were scaled to match the format of the training set data

- At the end of the function, <i>personal_data</i> vector was created. It contains all entered data in the same format as in the training set

- Function also returns unscaled <i>age, avg_glucose_level</i> and <i>bmi</i> of the user. Those values will be necessary later during Monte Carlo simulation

In [7]:
def profile():
    gender = get_generic_input('What is your biological gender? ', ('male', 'female'))
    gender = gender_mapping[gender]                                                                # Gender mapping to numeric value: male = 0 and female = 1
    age = get_integer_input('What is your age?', range_max=120)                                                                                          
    valid_work_types = input_employment_mapping.copy()                                             # User can't be a child if they are older than 17, so we are removing this option from the list of work types
    if age > 17:
        valid_work_types.pop('Children')
    work_type = get_generic_input("What is your work type?", valid_work_types)
    input_employment_mapping[work_type] = 1
    work_type = input_employment_mapping.values()
    hypertension = get_generic_input("Do you have hypertension?", ("yes", "no"))                   # Hypertension mapping to numeric value: yes = 1, no = 0
    hypertension = input_hypertension_mapping[hypertension]   
    heart_disease = get_generic_input("Do you have any heart disease?", ("yes", "no"))
    heart_disease = input_heart_disease_mapping[heart_disease]                                     # Heart disease mapping to numeric value: yes = 1, no = 0
    ever_married = get_generic_input("Have you ever been married?", ("yes", "no"))                 # Marital status mapping to numeric value: yes = 1, no = 0
    ever_married = input_ever_married_mapping[ever_married]
    residence_type = get_generic_input("What is your residence type?", ("urban", "rural"))         # Residence type mapping to numeric value: urban = 1, rural = 0
    residence_type = input_residence_mapping[residence_type]
    avg_glucose_level = get_float_input("Enter your average glucose level:", range_min = 0, range_max = 2500)
    bmi = get_float_input("Enter your BMI:", range_min = 0, range_max = 200)
    smoking_status = get_generic_input("What is your smoking status?", ("never smoked", "formerly smoked", "smokes"))
    input_smoking_mapping[smoking_status] = 1
    smoking_status = input_smoking_mapping.values()                                                  

    scaled_age = (age - scaler.mean_[0]) / np.sqrt(scaler.var_[0])                                 #Scaling all non-binary features to match the format of the training data
    scaled_avg_glucose_level = (avg_glucose_level - scaler.mean_[1]) / np.sqrt(scaler.var_[1])    
    scaled_bmi = (bmi - scaler.mean_[2]) / np.sqrt(scaler.var_[2])          

    personal_data = [gender, scaled_age, hypertension, heart_disease, ever_married, residence_type, scaled_avg_glucose_level, scaled_bmi, *work_type, *smoking_status]

    return personal_data, age, avg_glucose_level, bmi

- Variable <text style="color:#99ffcc">PROFILE</text> was defined to store the <i>personal_data</i> entered by user

- <i>start_age, start_glucose, start_bmi</i> were defined to store unstandarized user's data

- <text style="color:#99ffcc"> model.predict_proba </text> was used to calculate the probability that user is already in the group of patients with stroke (probability that user had a stroke until now)

In [ ]:
# User's profile
PROFILE, start_age, start_glucose, start_bmi = profile()                     # Random vector with user input data, which will be used for prediction
current_disctribution = model.predict_proba([PROFILE])[0][1]   

<h3 style="color:#99ffcc">Monte Carlo simulation</h3>

- There was assumed that BMI changes linearly with age and that average glucose level changes linearly with BMI

- The rate of change for BMI and glucose was calculated based on the dataset

- There were created $1000$ virtual patient's profiles for each age

- We assume that hypertension status, heart diseases status, matrimonial status, residence type, work type and smoking status don't change during lifespan

- Only variables that could change are age, average glucose level and BMI

- Random noise was added to BMI and glucose level changes for each virtual user to gain more realistic data

- Minimum BMI level was set to $10$ and minimum glucose level was set to $50$. Limitations were added to prevent creation of patients with unrealistic attributes

- The probability that user will have stroke until age $i$ is the arithmetic mean of probabilities calculated for each virtual patient

In [ ]:
#We are using the Monte Carlo simulation to predict at which age the probability of risk will be higher
def MonteCarlo():
    slope_bmi, intercept_bmi, _, _, std_err_bmi = linregress(dataset['age'], dataset['bmi'])                            # Change of BMI depending on age
    slope_gluco, intercept_gluco, _, _, std_err_gluco = linregress(dataset['bmi'], dataset['avg_glucose_level'])        # Change of glucose level depending on BMI
    future_risks = {} #Dictionary age:risk of stroke in that age

    for age_step in range(start_age + 1, 130):

        yearly_probs = []

        for j in range(1000): 

            # Simulating data for one virtual patient
            sim_bmi = intercept_bmi + (slope_bmi * age_step) + np.random.normal(0, std_err_bmi)
            sim_glucose = intercept_gluco + (slope_gluco * sim_bmi) + np.random.normal(0, std_err_gluco)
            
            # Ensuring simulated values are within realistic bounds. Minimum BMI = 10 and minimum glucose level = 50 
            sim_bmi = max(10, sim_bmi)
            sim_glucose = max(50, sim_glucose)

            # Scaling data to match the format of the training data
            scaled_vals = scaler.transform([[age_step, sim_glucose, sim_bmi]])
            s_age, s_gluco, s_bmi = scaled_vals[0]

            # Building random vector for a virtual patient
            sim_profile = [
                PROFILE[0],      #gender
                s_age,           #scaled age
                PROFILE[2],      #hypertension
                PROFILE[3],      #heart_disease
                PROFILE[4],      #ever_married
                PROFILE[5],      #residence_type
                s_gluco,         #scaled glucose
                s_bmi,           #scaled bmi
                *PROFILE[8:13],  #work_type 
                *PROFILE[13:16]  #smoking_status
            ]

            # Calculating 
            prob = model.predict_proba([sim_profile])[0][1]
            yearly_probs.append(prob)
        
        # The final future prognosis is a mean of yearly_probs
        future_risks[age_step] = np.mean(yearly_probs)

    return future_risks

- After Monte Carlo simulation we have the probability that user already had the stroke until age $i$, where $i$ is his current or future age

- In other words, we have an empirical distribution of the random variable $X(t)$ that says if user had the stroke until time $t$

- This empirical distribution will be stored as the dictionary in the form {age: probability_that_user_already_had_stroke}

In [ ]:
# All values of the empirical distribution for each age
past_risk = MonteCarlo()
DISTRIBUTION = {start_age:current_disctribution, **past_risk}

<h3 style="color:#99ffcc">Finding the probability density function</h3>

- We suspect that the random variable $X$ has logistic distribution (because logistic regression was used as ML model)

- First, logistic distribution function was defined. After that curve fit test was used to determine the distribution's coefficients

- Within <text style="color:#99ffcc">curve_fit()</text> function, bounds were set. This was necessary to avoid numerical instability of the hazard function defined later in the code, which was used to return infinite risk

- Simple statistic tests (r2, mse, rmse) were conducted to examine if the curve fit is accurate enough

In [ ]:
def Logistic_distribution(age,s,mu):
    return 1/(1+np.exp(-(age-mu)/s))

#We are running curve fit test for logistic distribution to find the probability density function
x = np.array(list(DISTRIBUTION.keys()))
y = np.array(list(DISTRIBUTION.values()))
parameters, _ = curve_fit(Logistic_distribution, x, y, bounds=([1e-5, 0], [100, 150]))        #Bounds are necessary - othervise hazard could be infinite
s, mu = parameters
prediction = Logistic_distribution(x, s, mu)
r2 = r2_score(y, prediction)
mse = mean_squared_error(y, prediction)
rmse = np.sqrt(mse)
print("Logistic distribution was assigned to your data. Here are the exact values of its parameters and features:\n")
print("s =", s)
print("mu =", mu)
print("R² =", r2)
print("MSE =", mse)
print("RMSE =", rmse)

- Finally, with calculated logistic distribution's coefficients, probability density function and hazard function can be defined

- Hazard function returns the current probability (risk) of stroke for the user, under condition that he survived without stroke until now

- Hazard stroke was plotted to show the risk in every age more neat


In [ ]:
def Log_probability_density_function(age):
    return (np.exp(-(age - mu) / (s))) / (s * (1 + np.exp(- (age-mu) / (s) ))**2)


def hazard_function(age):
    return 1 / (s * ( 1 + np.exp(-(age-mu)/s) ))

print("The current hazard of stroke for your age (", start_age, ") equals to:", hazard_function(start_age)*100, "%\n")

In [ ]:
#That proves that our random variable X (that says about the probability of stroke in exact time) has approximately logistic distribution
#Now we are calculating hazard
print("The current hazard of stroke for your age (", start_age, ") equals to:", hazard_function(start_age)*100, "%\n")
plt.plot(x, hazard_function(x)*100, label='Hazard Function')
plt.xlabel('Age')
plt.ylabel('Hazard of Stroke (%)')
plt.title('Hazard Function of Stroke by Age')
plt.legend()
plt.grid()
plt.show()